# Business Search with the Bright Data Python SDK

Find company records through `client.search.business.companies()` and work with the returned data directly. The Business category also provides `people()` for professional profiles.

This notebook contains two examples:

1. **Simple:** describe the companies you want in plain language and return three records.
2. **More detailed:** write structured filters, select output fields, fetch two pages, and combine the records without duplicate IDs.

Running both examples makes **at most three search requests**: one Instant request and up to two Ludicrous requests. Running a request cell again makes fresh requests. Search results and timings can differ between runs.

[Business Search documentation](https://docs.brightdata.com/products/business-search/introduction)

## Setup

Use Python 3.10 or later and a checkout of this repository that includes Business Search support. Install the checkout and a notebook kernel from the repository root:

```bash
python -m pip install -e . ipykernel
```

Open this notebook using that Python environment. Set `BRIGHTDATA_API_TOKEN` or `BRIGHTDATA_API_KEY` in your environment or the repository's `.env` file. The account behind the key needs Business Search access. The SDK reads the credentials; they are not printed or embedded in this notebook.

The examples use `await` directly in notebook cells. No Web Unlocker or SERP zone is needed.

In [ ]:
import json
from time import perf_counter

from dotenv import load_dotenv

from brightdata import BrightDataClient

load_dotenv()

CLIENT_OPTIONS = {
    "auto_create_zones": False,
    "validate_token": False,
}

## Example 1: Find companies with a plain-language query

Find **US software companies with 50 to 500 employees**.

- `mode="instant"` asks Bright Data to turn the sentence into a structured search.
- `view="summary"` requests a compact company record. Omitting `view` would return the identifier-only view.
- `limit=3` requests up to three records.

This cell makes one search request.

In [ ]:
async with BrightDataClient(**CLIENT_OPTIONS) as client:
    started = perf_counter()
    simple_page = await client.search.business.companies(
        query="US software companies with 50 to 500 employees",
        mode="instant",
        view="summary",
        limit=3,
    )
    simple_elapsed_ms = (perf_counter() - started) * 1000

### Read the records and request metadata

Each item in `page.documents` has a `bright_id` and a `data` dictionary. The next cell prints every returned record as plain, indented JSON, followed by request metadata. No rich JSON viewer is required. Individual fields can be missing.

This prints all fields returned by `view="summary"`. Change the request to `view="full"` if you want every stored field available for each record.

`matched` is a lower bound on matches reached by this search, not an exhaustive total. `coverage_percent` below 100 means the response is partial. `req_id` identifies the request for troubleshooting. An empty list is a valid result.

The elapsed time below includes the network, Bright Data processing, and SDK handling. It is one observation, not a benchmark.

In [ ]:
simple_companies = simple_page.documents
simple_metadata = {
    "req_id": simple_page.req_id,
    "source": simple_page.source,
    "returned_records": len(simple_page.documents),
    **simple_page.meta,
    "elapsed_ms": round(simple_elapsed_ms, 1),
}

print(f"Returned {len(simple_companies)} company records.")
if not simple_companies:
    print("No companies matched this search.")

for index, document in enumerate(simple_companies, start=1):
    name = document["data"].get("name") or "Name not returned"
    print(f"\nCompany {index}: {name}")
    print(json.dumps(document, indent=2, ensure_ascii=False))

print("\nRequest metadata")
print(json.dumps(simple_metadata, indent=2, ensure_ascii=False))

## Example 2: Build a company shortlist with filters and pagination

Write the conditions explicitly using `mode="ludicrous"`:

- The industry contains the word **software**.
- The headquarters country is **US**.
- The employee count recorded on LinkedIn is between **50 and 500**, inclusive.

`query` selects matching records; `view` selects the fields returned for each record. Here we request just five useful company fields.

Fetch at most two pages, at offsets 0 and 3, with up to three records each. Each page is a separate search against the index, so results can overlap or change. Keep the first record for each `bright_id` and record metadata for each page. Stop if a page is empty; do not use `matched` as a target for an unbounded paging loop.

In [ ]:
company_query = {
    "and": [
        {"text": {"industry": "software"}},
        {"equals": {"headquarters_country_code": "US"}},
        {"range": {"employees_in_linkedin": {">=": 50, "<=": 500}}},
    ]
}
company_fields = [
    "name",
    "website",
    "industry",
    "headquarters_country_code",
    "employees_in_linkedin",
]
page_size = 3
max_pages = 2

In [ ]:
companies_by_id = {}
page_metadata = []

async with BrightDataClient(**CLIENT_OPTIONS) as client:
    for page_number in range(max_pages):
        started = perf_counter()
        page = await client.search.business.companies(
            query=company_query,
            mode="ludicrous",
            view={"fields": company_fields},
            offset=page_number * page_size,
            limit=page_size,
        )
        elapsed_ms = (perf_counter() - started) * 1000
        page_metadata.append(
            {
                "req_id": page.req_id,
                "returned_records": len(page.documents),
                **page.meta,
                "elapsed_ms": round(elapsed_ms, 1),
            }
        )

        for document in page.documents:
            companies_by_id.setdefault(
                document["bright_id"],
                {
                    "bright_id": document["bright_id"],
                    **{field: document["data"].get(field) for field in company_fields},
                },
            )

        if not page.documents:
            break

company_shortlist = list(companies_by_id.values())

### Inspect the shortlist and the pages it came from

The next cell prints every company in the shortlist, followed by metadata for each page. Each record includes its ID and the five fields requested by the custom view. A repeated company ID appears only once in the shortlist.

The output is plain text, with no row, column, or string truncation applied by this notebook.

The page cap keeps this example small; it does not establish that every matching company was retrieved. The employee count is what the source records, not a guaranteed current headcount.

In [ ]:
print(f"Fetched {len(page_metadata)} pages; kept {len(company_shortlist)} unique companies.")
if not company_shortlist:
    print("No companies matched the structured search.")

for index, company in enumerate(company_shortlist, start=1):
    name = company.get("name") or "Name not returned"
    print(f"\nCompany {index}: {name}")
    print(json.dumps(company, indent=2, ensure_ascii=False))

print("\nPage metadata")
for index, metadata in enumerate(page_metadata, start=1):
    print(f"\nPage {index}")
    print(json.dumps(metadata, indent=2, ensure_ascii=False))

if any(item["coverage_percent"] < 100 for item in page_metadata):
    print("At least one page has partial index coverage; the shortlist is incomplete.")

## Adapt the examples

Change the country or employee range in `company_query` to search a different group of companies. Change `company_fields` to request a different set of supported output fields.

Keep pagination deliberate: increasing `max_pages` increases the number of API requests. These examples make no automatic retries.

The Business category also exposes `client.search.business.people()` for structured and plain-language queries. E-commerce is a separate category at `client.search.ecommerce.products()`, using structured queries with `mode="ludicrous"`.